# Student Exam Performance Analysis

The goal of this project is to investigate the factors influencing students' academic performance and to build a predictive model that estimates exam scores based on demographic and behavioral characteristics. Specifically, our goal is to understand which factors most affect student exam performance and how accurately exam scores can be predicted using these features.

In [1]:
import pandas as pd
import numpy as np

Import the dataset into a pandas data frame.

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "StudentPerformanceFactors.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "grandmaster07/student-exam-performance-dataset-analysis",
  file_path
)

df.head()

C:\Users\annak\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


-----

# Preprocessing

## Exploring the dataset

In [3]:
# Check dtypes of the columns and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6607 non-null   int64
 1   Attendance                  6607 non-null   int64
 2   Parental_Involvement        6607 non-null   str  
 3   Access_to_Resources         6607 non-null   str  
 4   Extracurricular_Activities  6607 non-null   str  
 5   Sleep_Hours                 6607 non-null   int64
 6   Previous_Scores             6607 non-null   int64
 7   Motivation_Level            6607 non-null   str  
 8   Internet_Access             6607 non-null   str  
 9   Tutoring_Sessions           6607 non-null   int64
 10  Family_Income               6607 non-null   str  
 11  Teacher_Quality             6529 non-null   str  
 12  School_Type                 6607 non-null   str  
 13  Peer_Influence              6607 non-null   str  
 14  Physical_Activity  

In [4]:
# Check the unique values in each categorical column
for col in df:
    if df[col].dtype == 'str':
        print(f"Unique values in column '{col}':")
        print(df[col].unique().tolist())
        print("========================================================")


Unique values in column 'Parental_Involvement':
['Low', 'Medium', 'High']
Unique values in column 'Access_to_Resources':
['High', 'Medium', 'Low']
Unique values in column 'Extracurricular_Activities':
['No', 'Yes']
Unique values in column 'Motivation_Level':
['Low', 'Medium', 'High']
Unique values in column 'Internet_Access':
['Yes', 'No']
Unique values in column 'Family_Income':
['Low', 'Medium', 'High']
Unique values in column 'Teacher_Quality':
['Medium', 'High', 'Low', nan]
Unique values in column 'School_Type':
['Public', 'Private']
Unique values in column 'Peer_Influence':
['Positive', 'Negative', 'Neutral']
Unique values in column 'Learning_Disabilities':
['No', 'Yes']
Unique values in column 'Parental_Education_Level':
['High School', 'College', 'Postgraduate', nan]
Unique values in column 'Distance_from_Home':
['Near', 'Moderate', 'Far', nan]
Unique values in column 'Gender':
['Male', 'Female']


In [5]:
# Numeric summary
df.describe()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


In [6]:
# Count missing values in each column
df.isnull().sum()

Hours_Studied                  0
Attendance                     0
Parental_Involvement           0
Access_to_Resources            0
Extracurricular_Activities     0
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               0
Internet_Access                0
Tutoring_Sessions              0
Family_Income                  0
Teacher_Quality               78
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_Education_Level      90
Distance_from_Home            67
Gender                         0
Exam_Score                     0
dtype: int64

## Handling the Missing Values

`Teacher_Quality`, `Parental_Education_Level`, and `Distance_from_Home` appear to be the only three columns containing any missing values. Based on the dtype check, all three also so happen to contain categorical values, so to handle these missing values we will fill them with the modes (most frequent values) for their respective columns. 

In [7]:
df['Teacher_Quality'] = df['Teacher_Quality'].fillna(df['Teacher_Quality'].mode()[0])
df['Parental_Education_Level'] = df['Parental_Education_Level'].fillna(df['Parental_Education_Level'].mode()[0])
df['Distance_from_Home'] = df['Distance_from_Home'].fillna(df['Distance_from_Home'].mode()[0])

# Verify that there are no more missing values
df.isnull().sum()

Hours_Studied                 0
Attendance                    0
Parental_Involvement          0
Access_to_Resources           0
Extracurricular_Activities    0
Sleep_Hours                   0
Previous_Scores               0
Motivation_Level              0
Internet_Access               0
Tutoring_Sessions             0
Family_Income                 0
Teacher_Quality               0
School_Type                   0
Peer_Influence                0
Physical_Activity             0
Learning_Disabilities         0
Parental_Education_Level      0
Distance_from_Home            0
Gender                        0
Exam_Score                    0
dtype: int64

## Encoding the Categorical Variables

Since machine learning models require numerical values, rather than strings, we need to encode the categorical columns accordingly. 

### Binary Columns

Columns `Extracurricular_Activities`, `Internet_Access`, and `Learning_Disabilities` have only two unique values 'Yes' and 'No', which can be represented in binary format/mapped directly to 1 and 0 respectively.

### Ordinal Columns

Columns `Parental_Involvement`, `Access_to_Resources`, `Motivation_Level`, `Family_Income`, and `Teacher_Quality` have just three unique values ('Low', 'Medium', and 'High'), which have a natural order and therefore can be replaced with integers 0, 1, and 2 respectively.

Column `Parental_Education_Level` contains three unique values 'High School', 'College', and 'Postgraduate', which can be encoded as 0, 1, and 2 respectively as well.

Column `Distance_from_Home` contains three unique values 'Near', 'Moderate', and 'Far', which can also be encoded as 0, 1, and 2 respectively.

### Nominal Columns

Finally, columns `School_Type` (Public vs Private), `Peer_Influence` (Positive vs Neutral vs Negative), and `Gender` (Male vs Female) have no natural order, therefore we cannot assign them with specific ordered numbers. Instead, we will apply One-Hot encoding using ```pd.get_dummies()```.

In [8]:
# NOTE: Only run this cell once to avoid re-encoding already encoded columns. If you need to re-run, please reload the original dataset first.

# Encode binary categorical variables
binary_mapping = {'Yes': 1, 'No': 0}
df['Extracurricular_Activities']    = df['Extracurricular_Activities'].map(binary_mapping)
df['Internet_Access']               = df['Internet_Access'].map(binary_mapping)
df['Learning_Disabilities']         = df['Learning_Disabilities'].map(binary_mapping)


# Encode ordinal categorical variables
levels_mapping = {'Low': 1, 'Medium': 2, 'High': 3}
df['Parental_Involvement'] = df['Parental_Involvement'].map(levels_mapping)
df['Access_to_Resources'] = df['Access_to_Resources'].map(levels_mapping)
df['Motivation_Level'] = df['Motivation_Level'].map(levels_mapping)
df['Family_Income'] = df['Family_Income'].map(levels_mapping)
df['Teacher_Quality'] = df['Teacher_Quality'].map(levels_mapping)

df['Parental_Education_Level'] = df['Parental_Education_Level'].map({
    'High School': 0,
    'College': 1,
    'Postgraduate': 2})

df['Distance_from_Home'] = df['Distance_from_Home'].map({
    'Near': 0,
    'Moderate': 1,
    'Far': 2})


# Encode nominal categorical variables using one-hot encoding
df = pd.get_dummies(df, columns=['School_Type', 'Peer_Influence', 'Gender'], drop_first=True)

In [9]:
# Check the dataframe info again to confirm changes
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Hours_Studied               6607 non-null   int64
 1   Attendance                  6607 non-null   int64
 2   Parental_Involvement        6607 non-null   int64
 3   Access_to_Resources         6607 non-null   int64
 4   Extracurricular_Activities  6607 non-null   int64
 5   Sleep_Hours                 6607 non-null   int64
 6   Previous_Scores             6607 non-null   int64
 7   Motivation_Level            6607 non-null   int64
 8   Internet_Access             6607 non-null   int64
 9   Tutoring_Sessions           6607 non-null   int64
 10  Family_Income               6607 non-null   int64
 11  Teacher_Quality             6607 non-null   int64
 12  Physical_Activity           6607 non-null   int64
 13  Learning_Disabilities       6607 non-null   int64
 14  Parental_Education_

## Defining Features and Target

Now we can split the dataframe into X (features) and y (target), where the target is `Exam_Score` and features are all the remaining columns.

In [10]:
X = df.drop(columns=['Exam_Score'])
y = df['Exam_Score']

## Splitting the Data Into Training and Testing Sets 

- 20% Testing
- 80% Training

In [11]:
# Install scikit-learn if needed:
# pip install scikit-learn
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=439)
# Set random_state for reproducibility, arbitrary choice of 439 to match course number

print(f"Train size: {len(X_train)}")
print(f"Test  size: {len(X_test)}")

Train size: 5285
Test  size: 1322


## Feature Scaling

We are planning to train a Linear Regression model, so it is useful to scale the numeric features so they don't dominate by magnitude. 

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [14]:
X_train.describe()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,School_Type_Public,Peer_Influence_Neutral,Peer_Influence_Positive,Gender_Male
count,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03,5.285000e+03
mean,7.932265e-17,3.737576e-16,-6.923926e-17,-3.361129e-17,-1.673842e-16,3.011572e-16,-6.991149e-17,-3.966133e-17,-6.722259e-17,6.587813e-17,-8.369212e-17,-2.218345e-17,6.923926e-17,-5.915588e-17,-6.722259e-18,8.201156e-17,7.260039e-17,3.630020e-17,-4.302246e-17,-3.562797e-17
std,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00,1.000095e+00
min,-3.187534e+00,-1.721426e+00,-1.568177e+00,-1.585593e+00,-1.221372e+00,-2.081102e+00,-1.735910e+00,-1.291928e+00,-3.504125e+00,-1.215404e+00,-1.056587e+00,-2.022050e+00,-2.893585e+00,-3.456389e-01,-8.977625e-01,-7.562702e-01,-1.516246e+00,-7.963662e-01,-8.261871e-01,-1.162766e+00
25%,-6.704080e-01,-8.586072e-01,-1.275694e-01,-1.555830e-01,-1.221372e+00,-7.090180e-01,-8.355471e-01,-1.291928e+00,2.853779e-01,-4.064104e-01,-1.056587e+00,-3.327464e-01,-9.412556e-01,-3.456389e-01,-8.977625e-01,-7.562702e-01,-1.516246e+00,-7.963662e-01,-8.261871e-01,-1.162766e+00
50%,8.255474e-04,4.212060e-03,-1.275694e-01,-1.555830e-01,8.187515e-01,-2.297623e-02,-4.442517e-03,1.525060e-01,2.853779e-01,-4.064104e-01,2.915691e-01,-3.327464e-01,3.490920e-02,-3.456389e-01,3.852756e-01,-7.562702e-01,6.595237e-01,-7.963662e-01,-8.261871e-01,8.600182e-01
75%,6.720591e-01,8.670313e-01,1.313038e+00,1.274427e+00,8.187515e-01,6.630655e-01,8.959207e-01,1.525060e-01,2.853779e-01,4.025836e-01,2.915691e-01,1.356557e+00,1.011074e+00,-3.456389e-01,3.852756e-01,7.423754e-01,6.595237e-01,1.255704e+00,1.210380e+00,8.600182e-01
max,4.028227e+00,1.729850e+00,1.313038e+00,1.274427e+00,8.187515e-01,2.035149e+00,1.727025e+00,1.596940e+00,2.853779e-01,4.447554e+00,1.639725e+00,1.356557e+00,2.963403e+00,2.893193e+00,1.668314e+00,2.241021e+00,6.595237e-01,1.255704e+00,1.210380e+00,8.600182e-01
